### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="south_africa_coronary_heart_disease",
    dataset_year="1983",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Kaggle", # Originally from http://statweb.stanford.edu/~tibs/ElemStatLearn/data.html
    original_dataset_source_download_link="https://www.kaggle.com/datasets/waalbannyantudre/south-african-heart-disease-dataset",
    download_description="""
We get the data from Kaggle, as it has the best curation and documentation.

kaggle datasets download waalbannyantudre/south-african-heart-disease-dataset && unzip south-african-heart-disease-dataset.zip && rm south-african-heart-disease-dataset.zip
mkdir -p local-data-warehouse/south_africa_coronary_heart_disease && mv SAHeart.csv local-data-warehouse/south_africa_coronary_heart_disease/
""",
    # References
    academic_reference_bibtex=r"""@article{rossouw1983coronary,
  title={Coronary risk factor screening in three rural communities. The CORIS baseline study.},
  author={Rossouw, JE and Du Plessis, JP and Benad{\'e}, AJ and Jordaan, PC and Kotze, JP and Jooste, PL and Ferreira, JJ},
  journal={South African medical journal= Suid-Afrikaanse tydskrif vir geneeskunde},
  volume={64},
  number={12},
  pages={430--436},
  year={1983}
}
""",
    academic_reference_bibtex_key="rossouw1983coronary",
    license="CC BY-SA 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from Kaggle.

- We remove the row id column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="chd",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="chd",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "SAHeart.csv")
print("Loaded data shape:", df.shape)

as_cat_type = ["famhist", "chd"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.drop(columns=["row.names"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()